# Factory Method: reservas de museo

## Introduccion

Un museo ofrece recorridos presenciales y recorridos virtuales. Ambos deben confirmar una reserva, pero cada tipo necesita crear una experiencia distinta. Factory Method permite que cada creador decida el producto concreto sin llenar el servicio de reservas de condicionales.

El problema es propio y representa una situacion extensible: el museo podria incorporar recorridos de realidad aumentada sin cambiar el flujo comun.

## Sin patron

La clase de reservas conoce todos los tipos de recorrido. Cada nuevo tipo obliga a modificar `crear_recorrido`, mezclando la decision de creacion con el flujo de confirmacion.

In [4]:
# Cada clase concreta representa una forma distinta de visitar el museo.
class RecorridoPresencial:
    def iniciar(self, visitante):
        return f"Guia recibe a {visitante} en la entrada"

class RecorridoVirtual:
    def iniciar(self, visitante):
        return f"Se envia enlace de videollamada a {visitante}"

# El condicional obliga a modificar esta funcion cuando aparece otro recorrido.
def confirmar_reserva(tipo, visitante):
    if tipo == "presencial":
        recorrido = RecorridoPresencial()
    elif tipo == "virtual":
        recorrido = RecorridoVirtual()
    else:
        raise ValueError("Tipo de recorrido no soportado")
    return recorrido.iniciar(visitante)

print(confirmar_reserva("presencial", "Mafe"))
print(confirmar_reserva("virtual", "Mafe"))

Guia recibe a Mafe en la entrada
Se envia enlace de videollamada a Mafe


## Con Factory Method

Roles: producto (`Recorrido`), productos concretos (`RecorridoPresencial`, `RecorridoVirtual`), creador (`ReservaMuseo`) y creadores concretos (`ReservaPresencial`, `ReservaVirtual`). El metodo de fabrica es `crear_recorrido`.

In [5]:
from abc import ABC, abstractmethod

# Producto abstracto: todos los recorridos deben poder iniciarse.
class Recorrido(ABC):
    @abstractmethod
    def iniciar(self, visitante):
        raise NotImplementedError

# Producto concreto para visitantes que asisten fisicamente.
class RecorridoPresencial(Recorrido):
    def iniciar(self, visitante):
        return f"Guia recibe a {visitante} en la entrada"

# Producto concreto para visitantes remotos.
class RecorridoVirtual(Recorrido):
    def iniciar(self, visitante):
        return f"Se envia enlace de videollamada a {visitante}"

# Creador abstracto: conserva el flujo y delega la creacion.
class ReservaMuseo(ABC):
    @abstractmethod
    def crear_recorrido(self):
        raise NotImplementedError

    def confirmar(self, visitante):
        # El cliente usa el flujo comun sin conocer la clase concreta.
        recorrido = self.crear_recorrido()
        return recorrido.iniciar(visitante)

# Cada subclase sobrescribe el Factory Method.
class ReservaPresencial(ReservaMuseo):
    def crear_recorrido(self):
        return RecorridoPresencial()

class ReservaVirtual(ReservaMuseo):
    def crear_recorrido(self):
        return RecorridoVirtual()

# Se pueden agregar creadores nuevos sin cambiar ReservaMuseo.
for reserva in (ReservaPresencial(), ReservaVirtual()):
    print(reserva.confirmar("Mafe"))

Guia recibe a Mafe en la entrada
Se envia enlace de videollamada a Mafe


## UML

```plantuml
@startuml
abstract class Recorrido
class RecorridoPresencial
class RecorridoVirtual
abstract class ReservaMuseo {
  +crear_recorrido()
  +confirmar(visitante)
}
class ReservaPresencial
class ReservaVirtual
Recorrido <|-- RecorridoPresencial
Recorrido <|-- RecorridoVirtual
ReservaMuseo <|-- ReservaPresencial
ReservaMuseo <|-- ReservaVirtual
ReservaMuseo ..> Recorrido : crea
@enduml
```

## Justificacion

Elegi Factory Method porque el flujo de confirmacion es comun y solo cambia el objeto que debe crearse. Adapter no resolveria una incompatibilidad de interfaces y Strategy se enfocaria en intercambiar algoritmos, no en delegar la creacion a subclases.